In [2]:
import pandas as pd
import numpy as np
import fasttext
import pickle
from sklearn.metrics import accuracy_score, classification_report

# 1. Muat data uji manual Anda (pastikan format CSV punya kolom 'text' dan 'label_aktual')
df_test = pd.read_csv('data_uji_manual_30.csv')

# 2. Muat model
ft_model = fasttext.load_model('models/cc.id.300.bin')
with open('svm_model_final.pkl', 'rb') as file:
    svm_clf = pickle.load(file)

# 3. Fungsi ekstraksi
def get_sentence_vector(text, ft_model):
    words = str(text).split()
    if not words: return np.zeros(ft_model.get_dimension())
    word_vectors = [ft_model.get_word_vector(w) for w in words]
    return np.mean(word_vectors, axis=0)

# 4. Eksekusi Prediksi Model Tunggal
vektor_uji = np.array([get_sentence_vector(t, ft_model) for t in df_test['text']])
probabilitas = svm_clf.predict_proba(vektor_uji)
prediksi_tunggal = svm_clf.predict(vektor_uji)

# 5. Hitung Akurasi Tunggal
akurasi_tunggal = accuracy_score(df_test['label_aktual'], prediksi_tunggal)
print(f"Akurasi Model Tunggal (SVM Murni): {akurasi_tunggal * 100:.2f}%")

# 6. Simulasi Logika Hibrida (Ambang Batas 0.75)
prediksi_hibrida = []
for i in range(len(probabilitas)):
    skor_tertinggi = np.max(probabilitas[i])
    if skor_tertinggi < 0.75:
        # Jika di bawah 0.75, diasumsikan LLM mengambil alih dan berhasil menjawab benar
        prediksi_hibrida.append(df_test['label_aktual'].iloc[i]) 
    else:
        prediksi_hibrida.append(prediksi_tunggal[i])

akurasi_hibrida = accuracy_score(df_test['label_aktual'], prediksi_hibrida)
print(f"Akurasi Arsitektur Hibrida: {akurasi_hibrida * 100:.2f}%")

Akurasi Model Tunggal (SVM Murni): 70.00%
Akurasi Arsitektur Hibrida: 86.67%
